# **Embedded Deep Learning System for the Detection of Cardiorespiratory Pathologies through Audio Analysis**
The major technological advances of recent decades are revolutionizing even the most traditional fields. The adoption of Artificial Intelligence makes it possible to streamline processes, facilitate decision-making, and automate industry. Following the COVID-19 pandemic, the use of AI techniques that enable rapid diagnosis of respiratory pathologies has become a key research focus. As respiratory diseases are one of the leading causes of death in today’s population, there is a clear need for non-invasive, low-cost methods that allow for early detection of pathologies and their prompt treatment.

Auscultation, which refers to the examination of internal body sounds using a stethoscope, involves a subjective diagnosis by the specialist. The presence of other internal body noises or the experience of the practitioner can affect the accuracy of disease detection. Additionally, it requires in-person examination of the patient, which, in the case of infectious diseases, poses a risk to both healthcare personnel and other workers or patients present in the medical facility.

The motivation behind the development of this Master’s Thesis lies in the need to address the deployment of a local respiratory pathology detection system that serves both as a support tool for qualified personnel in issuing diagnoses and as a teaching tool for students in training.

With the implementation of this tool, it becomes possible to apply advances in deep learning techniques within the field of sound analysis on a portable physical device. This significantly improves the accuracy of respiratory pathology diagnosis and reduces human error in auscultation-based assessments.

The following Notebook is structured as follows:

# **1.- Installation and Library Loading**
Below, the necessary libraries for the development of the project will be installed and loaded.

## **1.1.- Library Installation**
I proceed with the installation of the aforementioned libraries. Some libraries such as `multiprocessing`, `tempfile`, `pprint`, and `urllib` do not need to be installed, as they are part of Python’s standard library.

In [10]:
%%capture
pip install os-sys scipy librosa numpy matplotlib absl-py tensorflow tensorflow-io keras-tuner ipython

## **1.2.- Loading the Libraries**
I proceed to load the installed libraries.

In [ ]:
%%capture

# Standard library
import os, sys, gc, shutil, tempfile, urllib, pprint, multiprocessing

# Scientific computing
import numpy as np
import scipy
from scipy.signal import butter, lfilter

# Machine Learning / Deep Learning
import tensorflow as tf
import tensorflow_io as tfio
import keras_tuner
from sklearn.utils.class_weight import compute_class_weight

# Audio processing
import librosa
import librosa.display
import nlpaug.augmenter.audio as naa

# Visualization
import matplotlib.pyplot as plt
import cv2
import cmapy

# Misc
from IPython.display import Audio
import absl

# Config
tf.get_logger().propagate = False
pp = pprint.PrettyPrinter()

# **2.- Dataset Generation Process**
This section introduces the dataset generation process, which follows the methodology proposed in the RespireNet paper. The pipeline is composed of several key stages, including audio preprocessing, respiratory cycle segmentation based on annotations, feature extraction, and data generation and augmentation.

In summary, respiratory cycles are extracted from the labeled recordings and transformed into Mel spectrogram representations, which are later used as input features for the model.

Additionally, several auxiliary functions are defined to support the different stages of the process:

## **2.1.- Audio preprocessing**
This section defines a set of utility functions used to preprocess raw audio signals before feature extraction or model training. The preprocessing pipeline includes filtering, standardization, and length normalization.

### **Butterworth Bandpass Filter Design**
This function designs a Butterworth bandpass filter, which allows frequencies within a specific range to pass while attenuating others.

In [ ]:
def butter_bandpass(lowcut, highcut, fs, order=5):
    """
    Function to design a Butterworth bandpass filter.
    """
    # Design the Butterworth bandpass filter coefficients based on the specified lowcut, highcut frequencies, sampling rate (fs), and filter order.
    nyq = 0.5 * fs

    # Normalize the lowcut and highcut frequencies by the Nyquist frequency and compute the filter coefficients using the Butterworth filter design.
    low = lowcut / nyq
    high = highcut / nyq

    # Compute the Butterworth bandpass filter coefficients using the butter function from the scipy.signal library.
    b, a = butter(order, [low, high], btype='band')

    # Return the filter coefficients (b, a) for the designed Butterworth bandpass filter.
    return b, a

### **Apply Bandpass Filter**
This function applies the previously defined Butterworth filter to an audio signal.

In [ ]:
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    """
    Function to apply a Butterworth bandpass filter to the input data.
    """
    # Apply the Butterworth bandpass filter to the input data
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)

    # Return the filtered audio data
    return y

### **Audio Standardization**
This function standardizes the audio signal to have a consistent scale.

In [ ]:
def standardize_audio(audio):
    """
    Function to standardize the audio data by removing the mean and scaling to unit variance.
    """
    # Standardize the audio data by removing the mean and scaling to unit variance
    standardized_audio = (audio - -1.4100063212550931e-07) / 0.03260556890932612

    # Return the standardized audio data
    return standardized_audio

### **Length Adjustment**
This function ensures that all audio samples have the same length.

In [ ]:
def check_length_and_padding(audio, start_sample, target_length):
    """
    Function to ensure that the audio data has a specific target length by applying padding or truncation as needed.
    """
    # Check if the audio data is shorter than the target length and apply padding if necessary
    if len(audio) < target_length:
        # Randomly choose between constant padding and reflect padding to ensure the audio data reaches the target length
        if np.random.rand() > 0.5:
            padding = target_length - len(audio)
            segmented_audio = np.pad(audio, (0, padding), mode = 'constant')
        else:
            padding = target_length - len(audio)
            segmented_audio = np.pad(audio, (0, padding), mode = 'reflect')

    # Check if the audio data is longer than the target length and apply truncation if necessary
    elif len(audio) > target_length:
        segmented_audio = audio[:start_sample + target_length]

    # If the audio data is already of the target length, return it as is
    else:
        segmented_audio = audio

    # Return the audio data with the ensured target length
    return segmented_audio

## **2.2.- Feature Extraction**
This section converts preprocessed audio signals into Mel spectrogram images, which are commonly used as input features for deep learning models in audio classification tasks.

### **Save Features Extracted**
This function saves a spectrogram (or feature representation) as a .png image file. This function must be defined before `extract_features`, since it is called within it.

In [ ]:
def save_features(data, index_cycle, output_path, label, patient, type):
    """
    Function to generate and save the features (spectrograms) for each audio segment. The features are saved as .png images in the specified directory.
    """
    # Create the spectrogram filename based on the patient ID, index, and augmentation type
    path = f"{label}/{patient}_{index_cycle}_{type}.png"

    # Normalize the spectrogram data to the range [0, 255] and save it as a .png image
    img = cv2.normalize(data, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # Save the spectrogram image to the specified path
    cv2.imwrite(os.path.join(output_path, path), img)

    # Free memory after saving the image
    gc.collect()

### **Extract Features from Audio**
This function transforms an audio segment into a Mel spectrogram and prepares it for storage as an image.

In [ ]:
def extract_features(audio, output_path, label, patient, index_cycle, type):
    """
    Function to extract features (spectrograms) from the audio segments and save them as .png images in the specified directory.

    audio: the audio data.
    output_path: destination path for the features.
    label: label of the audio segment (e.g., 'Healthy', 'Unhealthy').
    patient: patient ID.
    index_cycle: index of the respiratory cycle.
    type: type of augmentation (e.g., 'Original', 'Augmented').
    """
    # Generate the Mel Spectrogram for the audio segment:
    spectrogram = librosa.feature.melspectrogram(y = audio, sr=4096, n_fft=256, hop_length=128, n_mels=64)
    spectrogram = librosa.power_to_db(spectrogram, ref=np.max)
    spectrogram = (spectrogram - spectrogram.min()) / (spectrogram.max() - spectrogram.min())
    spectrogram *= 255

    # Reshape the spectrogram to have a single channel (grayscale)
    spectrogram = spectrogram.reshape(spectrogram.shape[0], spectrogram.shape[1], 1)

    # Save the features for the original audio segment
    save_features(spectrogram, index_cycle, output_path, label, patient, type)

## **2.3.- Audio Segmentation and Feature Generation**
This function processes a full audio recording by dividing it into respiratory cycle segments, applying preprocessing, and generating corresponding spectrogram features.

In [ ]:
def divide_audio(duration, sample_rate, raw_audio, output_path, patient, cycles, cycles_train):
    """
    Function to divide the audio into segments based on the respiratory cycles and generate spectrograms for each segment.

    duration: target duration of the audio.
    sample_rate: sampling rate.
    raw_audio: raw audio data.
    output_path: destination path.
    patient: patient ID.
    cycles: respiratory cycles information.
    cycles_train: training respiratory cycles information.
    """
    # Define the target duration in number of samples
    target_length = int(duration * sample_rate)

    print(f"Processing patient: {patient} - Target length in samples: {target_length}")

    # Obtain the respiratory cycles for the specific patient
    cycles_filtered = cycles[np.isin(cycles[:, 1], patient)]

    # Auxiliar variable to keep track of the cycle number
    aux_index_cycle = 0

    # Now, for each respiratory cycle, I will extract the corresponding audio segment, apply the Butterworth filter, standardize it, and generate the spectrogram. 
    for index_cycle in cycles_filtered:
        # Define the start and end of the segment in terms of samples
        start = int(float(index_cycle[2]) * sample_rate)
        end = int(float(index_cycle[3]) * sample_rate)

        # Divide the audio segment corresponding to the respiratory cycle:
        segm = raw_audio[start:end]

        # Apply the Butterworth bandpass filter and standardize the audio segment
        segmented_audio = butter_bandpass_filter(segm, 50, 2000, sample_rate, order=5)
        segmented_audio = standardize_audio(segmented_audio)

        # Check wether the audio segment is shorter than the target duration. If so, we will apply padding to reach the desired length.
        segmented_audio = check_length_and_padding(segmented_audio, start, target_length)

        # Generate the Mel Spectrogram for the audio segment:
        extract_features(segmented_audio, output_path, label = index_cycle[6], patient = patient, index_cycle = aux_index_cycle, type = "Original")

        # Increment the cycle index for the next iteration
        aux_index_cycle += 1

    # Free memory after processing the audio file
    gc.collect()

## **2.4.- Generating Dataset**
This module is responsible for handling the end-to-end processing of audio files, from loading raw `.wav` recordings to generating spectrogram-based features.

### **Audio File Processing**
This function processes a single audio file, handling loading and segmentation, and triggering feature extraction.

In [ ]:
def process_file(file, input_path, output_path, length, cycles, cycles_train):
    """
    Function to process a single audio file. Includes: loading, segmenting, generating spectrograms and saving the results.
    """
    try:
        # Get unique record ID from the filename
        base_filename = os.path.basename(file).replace(".wav", "")
    
        # Obtain the information of the recording from the filename
        info_elements = base_filename.split("_")
        id_patient = info_elements[0] + "_" + info_elements[1] + "_" + info_elements[2] + "_" + info_elements[3] + "_" + info_elements[4]

        # Load the audio file using librosa
        raw_audio, sample_rate = librosa.load(os.path.join(input_path, file), sr=4096)

        # Proceed to audio segmentation
        divide_audio(duration = length, 
                     sample_rate = sample_rate, 
                     raw_audio = raw_audio, 
                     output_path = output_path, 
                     patient = id_patient, 
                     cycles = cycles, 
                     cycles_train = '')
        
    except Exception as e:
        print(f"Error while processing {input_path} - Patient {id_patient}: {e}")

### **Parallelization**
This function processes multiple audio files in parallel, enabling efficient handling of large datasets.

In [ ]:
def lectura_datos_parallel(input_path, output_path, length, cycles, cycles_train):
    """
    Function to read the data and process .wav files in parallel.
    """  
    # List of .wav files in the directory
    archivos_wav = [os.path.join(input_path, f) for f in os.listdir(input_path) if f.endswith(".wav")]

    # Load respiratory cycles
    cycles = np.load('/app/src/data/ciclos_respiratorios.npy')

    # Create tuples of parameters for each file
    argumentos = [(archivo, input_path, output_path, length, cycles, cycles_train) for archivo in archivos_wav]

    # Use multiprocessing to parallelize the processing of files
    with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
        pool.starmap(process_file, argumentos)

    return "Processing completed for all files."

## **2.5.- Augmentation**

### **CBA Augmentation**
This module implements a Class-Based Augmentation (CBA) strategy to address class imbalance in the dataset.

#### **Compute Dataset Imbalance**
This function calculates how many additional samples are needed per class to achieve a balanced dataset.

In [ ]:
def check_balanced_dataset(data_path):
    """
    Since each type of augmentation will be applied for a specific purpose, we need to calculate
    the number of augmented spectrograms that we need to generate for each class in order to balance the dataset.
    """
    # Obtain the list of spectrograms for each class.
    healthy = [os.path.join(data_path, f) for f in os.listdir(os.path.join(data_path, 'Healthy'))]
    crackle = [os.path.join(data_path, f) for f in os.listdir(os.path.join(data_path, 'Crackle'))]
    wheeze = [os.path.join(data_path, f) for f in os.listdir(os.path.join(data_path, 'Wheeze'))]
    both = [os.path.join(data_path, f) for f in os.listdir(os.path.join(data_path, 'Wheeze & Crackle'))]

    # Calculate the number of images that I need to generate to balance the classes, which will be equal to the max number of spectrograms for any class.
    num_max_reg = np.max([len(healthy), len(crackle), len(wheeze), len(both)])

    # Calculate the number of records that need to be augmented for each class.
    num_reg_aug_healthy = num_max_reg - len(healthy)
    num_reg_aug_crackle = num_max_reg - len(crackle)
    num_reg_aug_wheeze = num_max_reg - len(wheeze)
    num_reg_aug_both = num_max_reg - len(both)

    # Return the number of records that need to be augmented for each class to achieve a balanced dataset.
    return num_reg_aug_healthy, num_reg_aug_crackle, num_reg_aug_wheeze, num_reg_aug_both

#### **Select Respiratory Cycles**
This function selects two respiratory cycles based on a given class, following class-specific selection strategies.

In [ ]:
def select_random_respiratory_cycles(category, control_file_path = '/app/src/data/ciclos_respiratorios_train.npy'):
    """
    Function to select two random respiratory cycles.
    """
    # Load the file with the labeled cycles
    cycles = np.load(control_file_path)

    # Filter the respiratory cycles based on the specified class and a 
    # duration threshold of 4.5 seconds to ensure that only relevant cycles are selected for augmentation.
    cycles_healthy = cycles[ (cycles[:, 6] == 'Healthy') & (cycles[:, 11].astype(float) < 4.5)]
    cycles_crackle = cycles[ (cycles[:, 6] == 'Crackle') & (cycles[:, 11].astype(float) < 4.5)]
    cycles_wheeze= cycles[ (cycles[:, 6] == 'Wheeze') & (cycles[:, 11].astype(float) < 4.5)]
    cycles_both= cycles[ (cycles[:, 6] == 'Wheeze & Crackle') & (cycles[:, 11].astype(float) < 4.5)]

    # Depending on the specified class, select two random respiratory cycles from the corresponding filtered list of cycles.
    if category == 'Healthy':
        # Select two random respiratory cycles from the healthy class and return their indices and records for augmentation.
        # Since the healthy class needs to be augmented by only two healthy cycles, we only sample healthy ones.
        indices = np.random.choice(len(cycles_healthy), size=2, replace=False)
        regs = cycles_healthy[indices]

    # After that, for "Crackle" and "Wheeze" classes, we can either select two random cycles from the same class or a mix with one healthy cycle.
    elif category == 'Crackle':
        # With a 50% chance, select one random cycle from the crackle class and one random cycle from the healthy class to create a mixed augmentation, 
        # or select two random cycles from the crackle class for augmentation.
        if np.random.rand() < 0.5:
            # Select one random cycle with crackle and one random cycle from healthy class
            id_crackle = np.random.choice(len(cycles_crackle))
            id_healthy = np.random.choice(len(cycles_healthy))

            # Select the records corresponding to the chosen indices for augmentation
            registro_crackle = cycles_crackle[id_crackle]
            registro_healthy = cycles_healthy[id_healthy]

            # Combine the selected crackle and healthy records into a list for augmentation
            regs = [registro_crackle, registro_healthy]

            # Store the indices of the selected crackle and healthy cycles for reference during augmentation
            indices = [id_crackle, id_healthy]
        
        # As mentioned, the alternative is to select two random cycles from the crackle class for augmentation.
        else:
            # Select the two random cycles from crackle class.
            indices = np.random.choice(len(cycles_crackle), size=2, replace=False)
            regs = cycles_crackle[indices]

    # Same thing with "Wheeze" class, where we can either select two random cycles from the wheeze class or a mix with one healthy cycle for augmentation.
    elif category == 'Wheeze':
        # First, mix between Wheeze and Healthy class.
        if np.random.rand() < 0.5:
            # Select one random cycle with wheeze and one random cycle from healthy class
            id_wheeze = np.random.choice(len(cycles_wheeze))
            id_healthy = np.random.choice(len(cycles_healthy))

            # Select the records corresponding to the chosen indices for augmentation
            registro_wheeze = cycles_wheeze[id_wheeze]
            registro_healthy = cycles_healthy[id_healthy]

            # Combine the selected wheeze and healthy records into a list for augmentation
            regs = [registro_wheeze, registro_healthy]
            
            # Store the indices of the selected wheeze and healthy cycles for reference during augmentation
            indices = [id_wheeze, id_healthy]
        else:
            # Select two random cycles from the wheeze class
            indices = np.random.choice(len(cycles_wheeze), size=2, replace=False)
            regs = cycles_wheeze[indices]

    # Finally, for the "Wheeze & Crackle" class, we can generate a random combination of cycles from the crackle, wheeze, healthy, and both classes for augmentation,
    # or select two random cycles from the "Wheeze & Crackle" class for augmentation, depending on the available cycles in each class and a random choice.
    elif category == 'Wheeze & Crackle':
        
        # Generate a list of possible combinations.
        opciones = []

        # Depending on the availability of cycles in each class, add different combinations of classes to the list of options for augmentation.
        if len(cycles_both) > 0:
            if len(cycles_crackle) > 0:
                opciones.append('wheeze_crackle + crackle')
            if len(cycles_wheeze) > 0:
                opciones.append('wheeze_crackle + wheeze')
            if len(cycles_healthy) > 0:
                opciones.append('wheeze_crackle + healthy')
            if len(cycles_both) > 1:
                opciones.append('wheeze_crackle + wheeze_crackle')

        # If there are available options for augmentation, randomly select one of the combinations from the list of options and 
        # proceed with selecting the corresponding respiratory cycles for augmentation based on the chosen combination.
        if len(opciones) > 0:
            opcion = np.random.choice(opciones)

            # Depending on the randomly selected combination, select the corresponding respiratory cycles from the appropriate classes for augmentation 
            # and store their indices and records for reference during augmentation.
            if opcion == 'wheeze_crackle + crackle':
                # Mix between "Wheeze & Crackle" and "Crackle" classes by selecting one random cycle from each class for augmentation.
                id_both = np.random.choice(len(cycles_both))
                id_crackle = np.random.choice(len(cycles_crackle))

                # Select the records corresponding to the chosen indices for augmentation
                reg_wheeze_crackle = cycles_both[id_both]
                reg_crackle = cycles_crackle[id_crackle]

                # Combine the selected "Wheeze & Crackle" and "Crackle" records into a list for augmentation
                regs = [reg_wheeze_crackle, reg_crackle]

                # Store the indices of the selected "Wheeze & Crackle" and "Crackle" cycles for reference during augmentation
                indices = [id_both, id_crackle]

            # Same thing with "Wheeze & Crackle" and "Wheeze" classes, where we can mix between them by selecting one random cycle from each class for augmentation.
            elif opcion == 'wheeze_crackle + wheeze':
                # Mix between "Wheeze & Crackle" and "Wheeze" classes by selecting one random cycle from each class for augmentation.
                id_both = np.random.choice(len(cycles_both))
                id_wheeze = np.random.choice(len(cycles_wheeze))

                # Select the records corresponding to the chosen indices for augmentation
                reg_wheeze_crackle = cycles_both[id_both]
                reg_wheeze = cycles_wheeze[id_wheeze]

                # Combine the selected "Wheeze & Crackle" and "Wheeze" records into a list for augmentation
                regs = [reg_wheeze_crackle, reg_wheeze]

                # Store the indices of the selected "Wheeze & Crackle" and "Wheeze" cycles for reference during augmentation
                indices = [id_both, id_wheeze]

            # Same thing with "Wheeze & Crackle" and "Healthy" classes, where we can mix between them by selecting one random cycle from each class for augmentation.
            elif opcion == 'wheeze_crackle + healthy':
                # Mix between "Wheeze & Crackle" and "Healthy" classes by selecting one random cycle from each class for augmentation.
                id_both = np.random.choice(len(cycles_both))
                id_healthy = np.random.choice(len(cycles_healthy))

                # Select the records corresponding to the chosen indices for augmentation
                reg_wheeze_crackle = cycles_both[id_both]
                reg_healthy = cycles_healthy[id_healthy]

                # Combine the selected "Wheeze & Crackle" and "Healthy" records into a list for augmentation
                regs = [reg_wheeze_crackle, reg_healthy]

                # Store the indices of the selected "Wheeze & Crackle" and "Healthy" cycles for reference during augmentation
                indices = [id_both, id_healthy]

            elif opcion == 'wheeze_crackle + wheeze_crackle':
                # Randomly select two cycles from the "Wheeze & Crackle" class for augmentation.
                indices = np.random.choice(len(cycles_both), size=2, replace=False)
                regs = cycles_both[indices]
        else:
            raise ValueError("Not enough cycles available for 'Wheeze & Crackle' augmentation")
    
    # Finally, if an invalid category is specified, return an error message indicating that the category is not valid and providing the valid options for augmentation.
    else:
        return "Error: Invalid category specified. Please choose from 'Healthy', 'Crackle', 'Wheeze', or 'Wheeze & Crackle'."

    # Return the indices and records of the selected respiratory cycles for augmentation based on the specified class and random selection criteria.
    return indices, regs

#### **Perform CBA**
This function performs Class-Based Augmentation for a specific class by generating a new augmented sample.

In [ ]:
def CBA(category, duration, input_path, output_path):
    """
    Function to perform Class-Based Augmentation (CBA) for a specified class by selecting random respiratory cycles, applying augmentation techniques, 
    and generating augmented spectrograms for the selected class.
    """
    # Obtain the indices and records for the selected respiratory cycles
    indices, regs = select_random_respiratory_cycles(category)

    # Create list to store the audio segments.
    audios = []

    # Read the audio files and add them to the list.
    for cycle in regs:
        # Load the audio file corresponding to the current respiratory cycle using librosa, specifying the sampling rate (sr) for consistent processing.
        audio, sr = librosa.load(os.path.join(input_path, cycle[1]) + ".wav", sr = 4096)
        
        # Define start and end of the cycle.
        start = int(float(cycle[2]) * sr)
        end = int(float(cycle[3]) * sr)

        # Segment the audio.
        segment = audio[start:end]

        # Apply a Butterworth bandpass filter to the segmented audio.
        audio_segment = butter_bandpass_filter(segment, 50, 2000, 4096, order=5)
        
        # Apply standardization to the filtered audio segment.
        audio_segment = standardize_audio(audio_segment)

        # Add the processed audio segment to the list of audios for augmentation.
        audios.append(audio_segment)

    # Now, create the index for the concatenated audio
    index_generated = str(indices[0]) + "_" + str(indices[1])

    # Concat the two audio segments.
    concat_audio = np.concatenate((audios[0], audios[1]))

    # Defino la duración objetivo
    target_length = int(duration * sr)

    # Check wether the audio segment is shorter than the target duration. If so, we will apply padding to reach the desired length.
    # Why do we use 0 as the start sample? Mainly, because we want to keep the entire augmented audio segment.
    concat_audio_padded = check_length_and_padding(concat_audio, start_sample = 0, target_length = target_length)

    # Generate the Mel Spectrogram for the audio segment:
    extract_features(concat_audio_padded, output_path, label = category, patient = index_generated, index_cycle = index_generated, type = "CBA")

#### **Apply CBA for One Class**
This function applies CBA multiple times for a specific class.

In [ ]:
def apply_CBA_by_category(raw_audio_path = '/app/data/raw', output_path = '/app/data/processed/Train', duration = 6, number = 1, category = 'Default'):
    """
    Function to apply Class-Based Augmentation (CBA) for a specified class by calling the CBA function a certain number of times 
    to generate augmented spectrograms for the selected class.
    """
    # For the specified number of times, call the CBA function to perform Class-Based Augmentation (CBA) for the specified class and 
    # generate augmented spectrograms for that class.
    for i in range(number):
        # Aplico el aumento para la clase 'Healthy'
        CBA(category = category, duration = duration, input_path = raw_audio_path, output_path = output_path)

#### **Apply CBA to All Classes**
This function applies CBA to all classes, ensuring dataset balance.

In [ ]:
def apply_CBA(raw_audio_path = '/app/data/raw', output_path = '/app/data/processed/Train', duration = 6):
    """
    Function to apply Class-Based Augmentation (CBA) for all classes by calling the apply_CBA_by_category function for each class with the corresponding parameters.
    """
    # First, obtain the number of spectrograms that need to be augmented for each class.
    num_healthy, num_crackle, num_wheeze, num_both = check_balanced_dataset(output_path)

    # Now, apply Class-Based Augmentation (CBA) for each class by calling the apply_CBA_by_category function:
    apply_CBA_by_category(raw_audio_path, output_path, duration, num_healthy, category = 'Healthy')
    apply_CBA_by_category(raw_audio_path, output_path, duration, num_crackle, category = 'Crackle')
    apply_CBA_by_category(raw_audio_path, output_path, duration, num_wheeze, category = 'Wheeze')
    apply_CBA_by_category(raw_audio_path, output_path, duration, num_both, category = 'Wheeze & Crackle')

    return "CBA applied for all classes. Augmented spectrograms generated and saved to the specified output path."

# **3.- Dataset Loading**
This module handles the data preparation and training pipeline for the deep learning model.

## **3.1.- Labelling and Processing**
This section handles the extraction of labels from the dataset structure and the preprocessing of images before training.

### **Extract Label from Path**
This function extracts the class label from an image file path and converts it into a numerical representation.

In [ ]:
def get_label(file_path):
    """
    Function to extract the label from the file path, specifically from the folder.
    """
    # Split the file path into its components and extract the label string from the appropriate position (5th index).
    elements = tf.strings.split(file_path, os.path.sep)
    label_str = elements[5]

    # Create a lookup table to convert the label strings into numerical labels.
    keys = tf.constant(['Healthy', 'Crackle', 'Wheeze', 'Wheeze & Crackle'])
    values = tf.constant([0, 1, 2, 3], dtype=tf.int32)

    # Create a static hash table for the label lookup, with a default value of 4 for any unknown labels.
    table = tf.lookup.StaticHashTable( tf.lookup.KeyValueTensorInitializer(keys, values), default_value=4)

    # Use the lookup table to convert the label string into its corresponding numerical label.
    label = table.lookup(label_str)

    # Return the numerical label corresponding to the class of the image.
    return label

### **Process Image and Label**
This function loads and preprocesses an image, returning it along with its one-hot encoded label.

In [ ]:
def process_image(file_path):
    """
    Function to process an image file, including reading the image, decoding it, resizing it, and extracting the label.
    """
    # Extract the label from the file path using the get_label function.
    label = get_label(file_path)

    # Convert the label to a one-hot encoded vector with a depth of 4 (for 4 classes).
    label = tf.one_hot(label, depth=4)

    # Read the image file and decode it into a tensor with 3 color channels (RGB).
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3)

    # Set the shape of the image tensor to ensure it has 3 channels.
    img.set_shape([None, None, 3])

    # Resize the image to a fixed size of 64x193 pixels.
    img = tf.image.resize(img, [64, 193])

    # Return the processed image and its corresponding one-hot encoded label.
    return img, label

## **3.2.- Load Training and Testing Datasets**
This section is responsible for loading the training and testing datasets and preparing them for model training using an efficient data pipeline.

### **Load Datasets**
This function loads and prepares the training and testing datasets using TensorFlow’s data pipeline.

In [ ]:
def load_datasets(dir_dataset):
    """
    Function to load the training / testing datasets from the specified directories, process the images and labels, and prepare them for training.
    """
    # Load both training and testing datasets using tf.data.Dataset.list_files.
    train_dataset = tf.data.Dataset.list_files(os.path.join(dir_dataset, 'Train/*/*'), shuffle = False)
    test_dataset = tf.data.Dataset.list_files(os.path.join(dir_dataset, 'Test/*/*'), shuffle = False)

    # Map the process_image function to both datasets:
    train_dataset = train_dataset.map(process_image)
    test_dataset = test_dataset.map(process_image)

    # Make batches and shuffle them:
    train_dataset = train_dataset.shuffle(len(train_dataset)).batch(64, drop_remainder=True)
    test_dataset = test_dataset.shuffle(len(test_dataset)).batch(64, drop_remainder=True)

    # Return the prepared training and testing datasets.
    return train_dataset, test_dataset

### **Compute Class Weights**
This function computes class weights to address dataset imbalance during training.

In [ ]:
def get_class_weights(dataset):
    """
    Function to obtain the class weights for the 'CategoricalFocalLoss' loss function.
    """
    # Get labels from the dataset and concatenate them into a single array.
    y_train = np.concatenate([y.numpy() for x, y in dataset], axis=0)

    # Transform the one-hot encoded labels into integer labels by taking the argmax along the appropriate axis.
    y_train_int = np.argmax(y_train, axis=1)

    # Get the unique class labels from the integer labels to compute class weights.
    classes = np.unique(y_train_int)

    # Calculate the class weights (must return an array).
    class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_int)

    # Cast to a numpy array to ensure it is in the correct format for use in the loss function.
    class_weights_array = np.array(class_weights)

    # Return the dictionary and array of class weights.
    return dict(enumerate(class_weights)), class_weights_array

# **4.- Model Training**
This section defines the process of training the neural network using the prepared datasets.

In [ ]:
def train(model, train_dataset, val_dataset, epochs=100):
    """
    Function to train a given model using the provided training and validation datasets.

    model: the neural network model to be trained (e.g., a custom CNN, ResNet50, or VGG19).
    train_dataset: the dataset used for training the model.
    val_dataset: the dataset used for validating the model during training.
    class_weights: a dictionary mapping class indices to weights, used to handle class imbalance during training (optional).
    epochs: the number of epochs to train the model (default is 10).
    batch_size: the number of samples per batch during training (default is 32).
    callbacks: a list of Keras callbacks to be applied during training (optional).
    """
    # Obtain the class weights using the get_class_weights function, which computes the weights based on the test dataset.
    alpha_dict, alpha = get_class_weights(val_dataset)

    # Compile the model with the Adam optimizer, categorical cross-entropy loss function, and the defined metrics.
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
                  loss = 'CategoricalCrossentropy')

    # Train the model using the fit method.
    history = model.fit(train_dataset, epochs = epochs, validation_data = val_dataset, class_weight = alpha_dict, verbose = 2, callbacks=[ICBHI_Score_PrintingCallback(val_dataset)])

    # Save the model after training is complete.
    model.save(os.path.join('models', datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + '.h5'))

    # Return the training history, which contains information about the loss and metrics for each epoch.
    return history

# **5.- Model Architectures**
This section presents three different model architectures used for the classification task: a custom Convolutional Neural Network (CNN) and two transfer learning approaches based on pre-trained networks.

## **5.1.- Custom Convolutional Neural Network**
This function defines a custom Convolutional Neural Network (CNN) designed to classify spectrogram images into multiple respiratory condition classes.

In [ ]:
def create_custom_cnn(input_shape = (64, 193, 3), num_classes = 4):
    """
    Function to create a custom Convolutional Neural Network (CNN) architecture for classifying spectrogram images into different classes based on the input shape and number of classes.

    input_shape: the shape of the input data (e.g., (64, 64, 1) for grayscale spectrogram images).
    num_classes: the number of output classes for classification (e.g., 4 for Crackle, Wheeze, Wheeze & Crackle, Healthy).
    """
    model = models.Sequential([
    layers.Input(input_shape),
    
    # First convolutional block with 32 filters, kernel size of (3,3), batch normalization, ReLU activation, and max pooling with strides of (2,4).
    layers.Conv2D(filters = 64, kernel_size = (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'), 
    layers.MaxPooling2D(strides=(2,4)),

    # Second convolutional block with 64 filters, kernel size of (3,3), batch normalization, ReLU activation, and max pooling with strides of (2,4).
    layers.Conv2D(filters = 80, kernel_size = (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'), 
    layers.MaxPooling2D(strides=(3,3)),

    # Third convolutional block with 128 filters, kernel size of (5,5), batch normalization, ReLU activation, and max pooling with strides of (3,3).
    layers.Conv2D(filters = 128, kernel_size = (5,5), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'), 
    layers.MaxPooling2D(strides=(3,3)),

    # Now, I will flatten the output of the convolutional blocks and add fully connected layers with batch normalization, ReLU activation, and dropout for regularization.
    layers.BatchNormalization(),
    layers.Flatten(),

    # First fully connected layer with 1024 units, batch normalization, ReLU activation, and dropout with a rate of 0.4 for regularization.
    layers.Dense(1024, activation = 'relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    # Second fully connected layer with 4 units (corresponding to the number of classes) and softmax activation for multi-class classification.
    layers.Dense(1024, activation = 'relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    # Output layer with softmax activation for multi-class classification.
    layers.Dense(num_classes, activation = 'softmax')])

    # Finally, I will return the created model.
    return model

## **5.2.- ResNet-50**
This function defines a model based on transfer learning using the pre-trained ResNet50 architecture.

In [ ]:
def create_resnet_50(input_shape=(64, 193, 3), num_classes=4):
    # Check if the model is downloaded and if not, download it
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(64, 193, 3))

    # Freeze the base model layers to prevent them from being updated during training
    base_model.trainable = False

    # Add new layers on top for the specific task
    model = models.Sequential([
        # First, we use the pre-trained ResNet50 model
        base_model,

        # Then, flatten the output and add two blocks of fully connected layers.
        layers.Flatten(),

        # First FC (Fully Connected) with 1024 units, ReLU activation, and dropout for regularization.
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),

        # Second FC with 1024 units, ReLU activation, and dropout for regularization.
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),

        # Output layer with softmax activation for multi-class classification.
        layers.Dense(4, activation='softmax')])
    
    # Finally, we will return the created model.
    return model

## **5.3.- VGG-19**
This function defines a model based on transfer learning using the pre-trained VGG19 architecture.

In [ ]:
def create_vgg_19(input_shape=(64, 193, 3), num_classes=4):
    # Check if the model is downloaded and if not, download it
    base_model = VGG19(weights='imagenet', include_top=False, input_shape=(64, 193, 3))

    # Freeze the base model layers to prevent them from being updated during training
    base_model.trainable = False

    # Add new layers on top for the specific task
    model = models.Sequential([
        # First, we use the pre-trained VGG19 model
        base_model,

        # Then, flatten the output and add two blocks of fully connected layers.
        layers.Flatten(),

        #  First FC (Fully Connected) with 128 units, ReLU activation, and dropout for regularization.
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),

        # Second FC with 128 units, ReLU activation, and dropout for regularization.
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),

        # Output layer with softmax activation for multi-class classification.
        layers.Dense(4, activation='softmax')])

    # Finally, return the created model.
    return model